# Прогрессивное изучение трансформеров: от nanoGPT к Llama-3.2

В этом блокноте мы пройдем путь от простой академической модели до современного промышленного стандарта, изучая их архитектуру и избыточность весов (Low-Rank).

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoConfig
import matplotlib.pyplot as plt
import numpy as np
import inspect
import pandas as pd
from IPython.display import display

torch.manual_seed(42)

## Часть 1: nanoGPT (Академический стандарт)

nanoGPT — это минималистичная реализация GPT-2 от Андрея Карпатого. Она идеально подходит для понимания основ.

### Архитектура nanoGPT (Native Mermaid)

```mermaid
graph TD
    In[Input] --> Emb[Token + Pos Embedding]
    Emb --> B[Transformer Block]
    
    subgraph "nanoGPT Block"
        direction TB
        LN1[LayerNorm 1] --> MHA[Multi-Head Attention]
        MHA --> Res1((+))
        Res1 --> LN2[LayerNorm 2]
        LN2 --> MLP[FeedForward: ReLU]
        MLP --> Res2((+))
    end
    
    Res2 --> Head[LM Head]
    Head --> Out[Logits]
```

**Особенности:**
- **LayerNorm**: Классическая нормализация.
- **ReLU**: Простая активация в MLP.

In [ ]:
import sys
import os
# Добавляем корень проекта в пути
sys.path.append(os.path.abspath('../../'))
try:
    from src.model import GPTLanguageModel, Block
    print("--- Исходный код блока nanoGPT ---\n")
    print(inspect.getsource(Block))
except Exception as e:
    print(f"Ошибка импорта: {e}")

## Часть 2: Llama-3.2-1B (Промышленный стандарт)

Llama-3.2 — это вершина оптимизации. Несмотря на малый размер (1B), она использует все современные SOTA-фишки.

### Архитектура Llama-3.2 (Native Mermaid)

```mermaid
graph TD
    In[Input] --> Emb[Token Embedding]
    Emb --> B[Llama Block]
    
    subgraph "Llama-3 Block"
        direction TB
        RMS1[RMSNorm] --> GQA[Grouped-Query Attn + RoPE]
        GQA --> Res1((+))
        Res1 --> RMS2[RMSNorm]
        RMS2 --> SwiGLU[MLP: SwiGLU]
        SwiGLU --> Res2((+))
    end
    
    Res2 --> Head[Final RMSNorm + Head]
```

**Ключевые отличия от nanoGPT:**
- **RMSNorm**: Более эффективная нормализация (без вычитания среднего).
- **RoPE**: Rotary Position Embeddings вместо аддитивных.
- **SwiGLU**: Более сложная и эффективная активация в MLP.
- **Grouped-Query Attention (GQA)**: Оптимизация KV-кэша.

In [ ]:
model_id = "meta-llama/Llama-3.2-1B"
print(f"Загрузка конфигурации {model_id}...")
try:
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    print(f"Hidden size: {config.hidden_size}")
    print(f"Intermediate size (MLP): {config.intermediate_size}")

    with torch.device("meta"):
        llama_model = AutoModelForCausalLM.from_config(config)

    print("\nПример проекций в Attention:")
    print(llama_model.model.layers[0].self_attn.q_proj)
except Exception as e:
    print(f"Ошибка: {e}. Возможно нет доступа к весам на HF или отсутствует интернет.")

## Часть 3: Сравнительный SVD Анализ

Давайте сравним, как убывают сингулярные числа (спектр) у маленькой академической модели и у большой предобученной.

In [ ]:
def get_spectral_data(W):
    U, S, V = torch.svd(W)
    return S.detach().cpu().numpy()

print("Создание демонстрационных матриц для сравнения...")
# 1. Матрица nanoGPT (n_embd = 384)
W_nano = torch.randn(384, 384) * 0.02

# 2. Матрица Llama-1B (hidden_size = 2048)
W_llama = torch.randn(2048, 2048) * 0.02 # Эмуляция

s_nano = get_spectral_data(W_nano)
s_llama = get_spectral_data(W_llama)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(s_nano / s_nano[0])
plt.title("Спектр nanoGPT (normalized)")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(s_llama / s_llama[0])
plt.title("Спектр Llama-1B (normalized)")
plt.grid(True)
plt.show()

print("Если спектр падает быстро — матрица легко сжимается (low-rank).")